# GÜN 33: Retrieval-Augmented Generation (RAG) ve Kaynaklı Cevap Üretimi (Attributed Generation)
## Staj Defteri: Yaprak 65 & 66 | Merinos Halı Sanayi A.Ş. — Endüstriyel Yapay Zekâ Stajı

---

> ### **ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR**
> **Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)**  
> Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlı olarak paylaşılmıştır.  
> Yazarın açık yazılı izni olmaksızın kopyalanamaz, çoğaltılamaz, dağıtılamaz veya ticari/ticari olmayan projelerde kullanılamaz.  
> İzin talepleri için: GitHub @seydivakkas  
> **Lisans Rozeti:** `https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square`

---

### **Staj Defteri Konu ve Kapsam Özeti**
* **Yaprak 65:** Endüstriyel RAG Mimarisi, Kapalı Dünya İlkesi (Closed-World Assumption), Getirilen Parçaların `[S1]`, `[S2]`, `[S3]` Formatında Etiketlenmesi ve Context Enjeksiyonu.
* **Yaprak 66:** Atıf ve İddia Doğrulama Motoru (Citation Verifier), Metinsel Sadakat (Faithfulness) Skoru, Halüsinasyon Tespiti, Retrieval Failure ile Generation Hallucination Hata Ayrımı ve Dürüst Reddetme (Abstention).



## 1. Teorik Çerçeve: Endüstriyel Dokuma Fabrikasında RAG Güvenliği

Klasik üretken yapay zekâ (LLM) modelleri, parametrik hafızalarında tuttukları bilgilerle cevap üretirler. Ancak **Merinos Halı Sanayi A.Ş.** gibi yüksek hassasiyetli bir endüstriyel üretim tesisinde parametrik hafızaya güvenilemez:
1. **Halüsinasyon Riski:** Model, olmayan bir arıza kodu uydurabilir veya yanlış bir hidrolik basınç değeri (örn. 6 bar yerine 14 bar) önererek tezgâh kırımına yol açabilir.
2. **Denetlenebilirlik (Traceability):** Operatör veya bakım mühendisi, verilen her talimatın hangi fabrika SOP (Standart Operasyon Prosedürü) belgesine dayandığını anında görebilmelidir.
3. **Dürüst Reddetme (Honest Abstention):** Dokümanda yer almayan bir soru sorulduğunda model tahmin yürütmemeli; kesin bir dille *"Verilen fabrika dokümanlarında bu konuyla ilgili yeterli bilgi bulunmamaktadır."* diyebilmelidir.

### **RAG İş Akışı ve Context Formatlama Mimarisi:**
$$\text{Query } q \xrightarrow{\text{Hibrit Arama (BM25 + Dense)}} \mathcal{C}_k = \{c_1, c_2, \dots, c_k\}$$

Her parça tekil bir kaynak etiketiyle dönüştürülür:
$$c_i \implies [\text{S}i] \quad \text{Başlık: } \text{title}_i \mid \text{Bölüm: } \text{sec}_i \mid \text{Metin: } \text{text}_i$$

Üretilen cevap iddialara bölünür:
$$\text{Cevap } A = \{a_1, a_2, \dots, a_m\}$$

Her $a_j$ iddiası için kaynak sadakati:
$$\text{Faithfulness}(a_j, S_k) = \frac{|\text{Kelimeler}(a_j) \cap \text{Kelimeler}(S_k)|}{|\text{Kelimeler}(a_j)|}$$



In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Day 33 - RAG ve Atıflı (Attributed) Yanıt Üretimi Hazır.")
# Merinos Endüstriyel Teknik Dokümantasyon Külliyatı (Bellek İçi Sentetik Veri)
MERINOS_DOCS = [
    {
        "doc_id": "DOC-001",
        "title": "SOP-401: Ana Tahrik Motoru Termal Koruma ve Aşırı Isınma",
        "text": "Vandewiele jakarlı dokuma tezgâhlarında ana tahrik motoru gövde sıcaklığı 85°C üzerine çıktığında termal koruma rölesi E-401 arıza kodunu tetikler ve tezgâhı durdurur. Operatör fan ızgaralarını temizlemeli, yağlama basıncını kontrol etmeli (min 3.5 bar) ve motorun 15 dakika soğumasını beklemelidir."
    },
    {
        "doc_id": "DOC-002",
        "title": "SOP-102: Çözgü ve Atkı İpliği Gerginlik Kontrolü",
        "text": "Akrilik ve polipropilen iplik bobinlerinde çözgü gerginliği 35 ile 45 cN aralığında sabit tutulmalıdır. Gerginlik 55 cN üzerine çıktığında atkı kopuş sensörü tezgâhı 0.2 saniyede durdurur. Operatör tansiyon yaylarını kontrol etmeli ve cağlık gergi ağırlıklarını yeniden ayarlamalıdır."
    },
    {
        "doc_id": "DOC-003",
        "title": "SOP-205: Rulman Yağlama ve Periyodik Bakım",
        "text": "Ana mil ve armür rulmanları her 500 çalışma saatinde bir ISO VG 220 sentetik sanayi yağı ile yağlanmalıdır. Yetersiz yağlama rulman titreşimini 4.5 mm/s üzerine çıkarır ve aşınmaya yol açar. Otomatik yağlama pompası basıncı 3.5 bar altına düşerse tezgâh kilitlenir."
    },
    {
        "doc_id": "DOC-004",
        "title": "SOP-308: Jakar Tarak ve Kanca Değişimi",
        "text": "Hereke ve Uşak desenlerinde tarak boşluğu 0.8 mm tolerans dahilinde kalmalıdır. Jakar kancalarının aşınması desen bozulmasına ve yüzey ilme atlama hatasına neden olur. Her 2000 saatte kanca yay gerilim testi yapılmalı ve deforme kancalar yenilenmelidir."
    },
    {
        "doc_id": "DOC-005",
        "title": "SOP-510: Dokuma Salonu İş Sağlığı ve Güvenliği",
        "text": "Dokuma salonunda çelik burunlu iş ayakkabısı ve kulak tıkacı takılması zorunludur. Tezgâh çalışır durumdayken acil stop butonları kesinlikle baypas edilemez ve koruyucu kapaklar sökülemez. Bakım öncesi tezgâh panosundan ana şalter kilitlenmelidir (LOTO)."
    }
]



✅ Tüm RAG ve Doğrulama modülleri başarıyla yüklendi.


## 2. Fabrika Bilgi Tabanı ve Hibrit Arama Motorunun Başlatılması


In [2]:
# Bağlam Kurucu (Context Builder) ve Atıflı Üretim Simülasyonu
query = "E-401 arıza kodu neden oluşur ve operatör ne yapmalıdır?"
retrieved_doc = MERINOS_DOCS[0]

# Üretilen Atıflı Yanıt
attributed_answer = (
    "Ana tahrik motor gövde sıcaklığı 85°C üzerine çıktığında termal koruma rölesi "
    "E-401 arıza kodunu tetikler ve tezgâhı durdurur [DOC-001]. "
    "Operatör fan ızgaralarını temizlemeli, yağlama basıncını kontrol etmeli (min 3.5 bar) "
    "ve motorun 15 dakika soğumasını beklemelidir [DOC-001]."
)

# Halüsinasyonlu Yanıt (Karşılaştırma için)
hallucinated_answer = (
    "E-401 kodu motor aşırı ısındığında oluşur [DOC-001]. "
    "Operatör motor gövdesine acilen soğuk su dökerek hızla soğutmalıdır."  # Tehlikeli Halüsinasyon!
)

def evaluate_faithfulness(answer, context):
    claims = [c.strip() for c in answer.split(". ") if c.strip()]
    grounded = 0
    for claim in claims:
        # Bağlamda geçen anahtar kelimeleri doğrula
        words = [w for w in claim.lower().split() if len(w) > 4]
        if sum(1 for w in words if w in context.lower()) >= len(words) * 0.5:
            grounded += 1
    return grounded / max(len(claims), 1)

f_safe = evaluate_faithfulness(attributed_answer, retrieved_doc["text"])
f_halluc = evaluate_faithfulness(hallucinated_answer, retrieved_doc["text"])

print(f"Sorgu: '{query}'")
print(f"Güvenli Yanıt Sadakat Skoru       : %{f_safe * 100:.1f}")
print(f"Halüsinasyonlu Yanıt Sadakat Skoru: %{f_halluc * 100:.1f}")



W0924 08:11:13.640000 30220 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Toplam Doküman Sayısı: 3
Toplam İndekslenen Parça: 10


## 3. Context Builder: Parçaların [S1], [S2], [S3] Kaynak Etiketleriyle Paketlenmesi


In [3]:
# Atıflı Üretim ve Sadakat Değerlendirme Paneli
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Attributed Generation & Faithfulness Guardrail (Day 33)", fontsize=13, fontweight="bold")

models = ["Güvenli RAG (Atıflı)", "Halüsinasyonlu Yanıt"]
scores = [f_safe * 100, f_halluc * 100]
colors = ["#2ca02c", "#d62728"]

axes[0].bar(models, scores, color=colors)
axes[0].set_ylim(0, 115)
axes[0].set_title("1. Sadakat / Doğrulanabilirlik Skoru (%)")
axes[0].set_ylabel("Sadakat %")

# İddia Denetimi Dağılımı
axes[1].pie([2, 1], labels=["Kanıtlanmış İddia (2)", "Doğrulanamayan İddia (1)"], 
            colors=["#2ca02c", "#d62728"], autopct="%1.1f%%", startangle=140)
axes[1].set_title("2. Halüsinasyonlu Yanıt İddia Dağılımı")

plt.tight_layout()
plt.show()



--- OLUŞTURULAN FORMATLI CONTEXT METNİ ---
--- BAŞLANGIÇ: TEKNİK FABRİKA DOKÜMANLARI (CONTEXT) ---

[S1] Doküman: merinos_weaving_sop.pdf | Bölüm: 2. Bolum: Ariza Kodlari ve Acil Durdurma
Hiyerarşi: Sayfa 1: Merinos Weaving SOP > Sayfa 2 > 2. Bolum: Ariza Kodlari ve Acil Durdurma
Metin:
2. Bolum: Ariza Kodlari ve Acil Durdurma
E-401 Ariza Kodu: Ana tahrik motoru asinmasi veya asiri isinmasi durumunda verilir.
E-401 goruldugunde operator derhal kirmizi acil durdurma butonuna basmali ve motor fanini temizlemelidir.
E-108 Ariza Kodu: Mekik iplik rezerv sensoru bos kaldi hatasidir.
E-256 Ariza Kodu: Cozgu cerceve kilit mekanizmasi basinci 6 bar altina dustugunde olusur.

[S2] Doküman: merinos_quality_standards.docx | Bölüm: BÖLÜM 1: İplik Mukavemet ve Tansiyon Basıncı
Hiyerarşi: Merinos Halı Kalite Güvence ve İplik Mukavemet Standartları > Sayfa 1 > BÖLÜM 1: İplik Mukavemet ve Tansiyon Basıncı
Metin:
BÖLÜM 1: İplik Mukavemet ve Tansiyon Basıncı

Dokuma sırasında akrilik ve polipropilen ipl

## 4. Attributed Generation ve Citation Verifier ile Güvenlik Doğrulaması


## 5. Güvenlik Deneyi: Simüle Edilmiş Halüsinasyon Tespiti

Eğer model dış dünyadan uydurma bir parametre veya tezgâh modeli eklerse, CitationVerifier bunu tespit edip alarm üretebilir mi?


## 6. Uçtan Uca RAG Değerlendirmesi: 15 Altın Sorgu ve Hata Sınıflandırması

Sorgular 5 kategoriyi kapsar:
1. `ERROR_CODE`: Tezgâh arıza kodları (`E-401`, `E-108`, `E-256`)
2. `WEAVING_SPEC`: Jakarlı dokuma toleransları, sarı ikaz lambaları, atkı sıklığı
3. `FINISHING_PROCESS`: Buharlı fikse, traşlama bıçak yüksekliği, overlok dikiş mukavemeti
4. `QUALITY_TOLERANCE`: Nem, sıcaklık şartlandırması ve renk sürekliliği
5. `OUT_OF_DOMAIN`: Negatif kontrol (yemekhane menüsü, personel servisi) $\rightarrow$ Dürüst Çekilme beklenir.


## 7. 4 Panelli Endüstriyel RAG Değerlendirme Paneli

1. Metinsel Sadakat (Faithfulness) Dağılımı
2. Atıf Başarımı (Citation Precision & Recall)
3. RAG Hata Ayrımı (Success vs. Retrieval Failure vs. Hallucination vs. Abstention)
4. Uçtan Uca Yanıt Süresi (Latency)


## 8. Endüstriyel Mühendislik Çıkarımları ve Staj Özeti

Bu çalışmada elde edilen kritik mühendislik bulguları:
1. **Retrieval Failure vs. Generation Hallucination Ayrımı:** Eğer arama motoru doğru parçayı Top-3 içine getiremediyse (`Hit@3 = False`), modelin eksik veya yanlış cevap üretmesi bir *üretim (generation)* hatası değil, bir *arama (retrieval)* hatasıdır. Hatanın kaynağını bilmek, sorunun chunking boyutunda mı yoksa LLM isteminde mi olduğunu izole eder.
2. **Citation Verifier Güvenlik Mandası:** `[S1]`, `[S2]` kaynak etiketleri sayesinde her iddia anında doğrulanabilmektedir. Simülasyon deneyimiz, %70 sadakat eşiğinin altında kalan uydurma teknik terimleri anında tespit ettiğini kanıtlamıştır.
3. **Dürüst Çekilme (Honest Abstention):** `RAG_Q15` negatif kontrol sorgusunda sistem kapalı dünya varsayımına sadık kalmış ve fabrika dışı soruyu başarıyla reddetmiştir. Bu, üretim sahasında sıfır hata prensibi için vazgeçilmezdir.

---
**Rapor Hazırlayan:** Seydi Eryılmaz  
**Görevi:** Merinos Halı Sanayi A.Ş. Yapay Zekâ & Otomasyon Stajyeri  
**Telif Hakkı (c) 2026 Seydi Eryılmaz — Tüm Hakları Saklıdır.**

